In [1]:
import pandas as pd
import json
import math
import numpy as np
from sentence_transformers import SentenceTransformer

In [2]:
# Define file paths
PASSAGE_COLLECTION_PATH = "../data/ms_marco/raw/collection.tsv"
QUERY_RELEVANCE_PATH = "../data/ms_marco/raw/qrels_dev.tsv"
QUERY_PATH = "../data/ms_marco/raw/queries_dev.tsv"

# Read in the dataframes
passage_collection_df = pd.read_csv(PASSAGE_COLLECTION_PATH, sep="\t", names=["passage_id", "passage_text"])
relevance_df = pd.read_csv(QUERY_RELEVANCE_PATH, sep="\t", names=["query_id", "query_relevance", "passage_id", "passage_relevance"])
query_df = pd.read_csv(QUERY_PATH, sep="\t", names=["query_id", "query_text"])

In [3]:
# EDA
print("Passage Collection DataFrame Info:")
print(passage_collection_df.info())
print(passage_collection_df.head())

Passage Collection DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8841823 entries, 0 to 8841822
Data columns (total 2 columns):
 #   Column        Dtype 
---  ------        ----- 
 0   passage_id    int64 
 1   passage_text  object
dtypes: int64(1), object(1)
memory usage: 134.9+ MB
None
   passage_id                                       passage_text
0           0  The presence of communication amid scientific ...
1           1  The Manhattan Project and its atomic bomb help...
2           2  Essay on The Manhattan Project - The Manhattan...
3           3  The Manhattan Project was the name for a proje...
4           4  versions of each volume as well as complementa...


In [4]:
print("Relevance DataFrame Info:")
print(relevance_df.info())
print(relevance_df.head())

Relevance DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59273 entries, 0 to 59272
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   query_id           59273 non-null  int64
 1   query_relevance    59273 non-null  int64
 2   passage_id         59273 non-null  int64
 3   passage_relevance  59273 non-null  int64
dtypes: int64(4)
memory usage: 1.8 MB
None
   query_id  query_relevance  passage_id  passage_relevance
0   1102432                0     2026790                  1
1   1102431                0     7066866                  1
2   1102431                0     7066867                  1
3   1090282                0     7066900                  1
4     39449                0     7066905                  1


In [5]:
print("Query DataFrame Info:")
print(query_df.info())
print(query_df.head())

Query DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101093 entries, 0 to 101092
Data columns (total 2 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   query_id    101093 non-null  int64 
 1   query_text  101093 non-null  object
dtypes: int64(1), object(1)
memory usage: 1.5+ MB
None
   query_id                      query_text
0   1048578  cost of endless pools/swim spa
1   1048579                    what is pcnt
2   1048580               what is pcb waste
3   1048581                   what is pbis?
4   1048582                  what is paysky


In [6]:

# Check the if the `query_id` in the relevance dataframes are unique (i.e. if the len of unique values is the same as the len of the dataframe)
# If they are not unique, means that there are multiple passages relevant to the same query
relevance_df_unique_queries = relevance_df["query_id"].nunique()
relevance_df_total_queries = len(relevance_df)
print(f"Relevance DataFrame:")
print(f"Unique Queries: {relevance_df_unique_queries}, Total Queries: {relevance_df_total_queries}")

Relevance DataFrame:
Unique Queries: 55578, Total Queries: 59273


In [7]:
# Check for queries that are not in relevance set
query_df_unique_qids = set(query_df["query_id"].unique())
relevance_df_unique_qids = set(relevance_df["query_id"].unique())
qids_missing_in_relevance_df = query_df_unique_qids - relevance_df_unique_qids
print(f"Number of query IDs in query set not in relevance set: {len(qids_missing_in_relevance_df)}")

Number of query IDs in query set not in relevance set: 45515


In [8]:
# Check for passages that are in relevance set but not in passage collection
passage_ids_in_collection = set(passage_collection_df["passage_id"].unique())
passage_ids_in_relevance = set(relevance_df["passage_id"].unique())
missing_passages_in_collection = passage_ids_in_relevance - passage_ids_in_collection
print(f"Number of passage IDs in relevance set not in passage collection: {len(missing_passages_in_collection)}")

Number of passage IDs in relevance set not in passage collection: 0


In [9]:
# Filter for queries in query set that exist in relevance set
filtered_query_df = query_df[query_df["query_id"].isin(relevance_df_unique_qids)]
print(f"Filtered Query DataFrame Length (only queries present in relevance set):")
print(len(filtered_query_df))

Filtered Query DataFrame Length (only queries present in relevance set):
55578


In [10]:
# Create a mapping of query to relevant passages from `relevance_df` for fast lookup
qids_to_relevant_pids = (relevance_df.groupby("query_id")["passage_id"]
                            .apply(lambda x: [str(pid) for pid in x])
                            .to_dict())

# For each query in dev set, we want to format it as:
# {
#   "qid": "1234",
#   "query": "how to change iphone battery",
#   "qrels": ["201", "55689"]  // from qrels.dev
# }

queries_with_qrels = []

for _, row in filtered_query_df.iterrows():
    qid = row["query_id"]
    query_text = row["query_text"]
    qrels = qids_to_relevant_pids.get(qid, [])
    queries_with_qrels.append({
        "qid": str(qid),
        "query": query_text,
        "qrels": qrels
    })

In [11]:
# Check how a line looks like
print(queries_with_qrels[0])

{'qid': '1048578', 'query': 'cost of endless pools/swim spa', 'qrels': ['7187234']}


In [12]:
# Check the maximum number of relevant passages for any query
max_relevant_passages = max(len(item["qrels"]) for item in queries_with_qrels)
print(f"Maximum number of relevant passages for any query: {max_relevant_passages}")

Maximum number of relevant passages for any query: 6


In [13]:
QUERIES_WITH_QRELS_OUTPUT_PATH = "../data/ms_marco/processed/queries_with_qrels.jsonl"

# Save formatted data to be JSONL
with open(QUERIES_WITH_QRELS_OUTPUT_PATH, "w") as f:
    for item in queries_with_qrels:
        f.write(json.dumps(item) + "\n")

print(f"Saved formatted queries with qrels to {QUERIES_WITH_QRELS_OUTPUT_PATH}")

Saved formatted queries with qrels to ../data/ms_marco/processed/queries_with_qrels.jsonl


In [14]:
# Get the set of passage IDs that are not present in the relevance set
all_passage_ids = set(passage_collection_df["passage_id"].unique())
relevant_passage_ids = set(relevance_df["passage_id"].unique())
non_relevant_passage_ids = all_passage_ids - relevant_passage_ids
print(f"Total unique relevant passage IDs: {len(relevant_passage_ids)}")
print(f"Total unique non-relevant passage IDs: {len(non_relevant_passage_ids)}")

Total unique relevant passage IDs: 59096
Total unique non-relevant passage IDs: 8782727


In [15]:
# Randomly sample 150,000 non-relevant passage IDs 
# Set seed for reproducibility
np.random.seed(42)
sampled_non_relevant_passage_ids = set(
    np.random.choice(list(non_relevant_passage_ids), size=150000, replace=False)
)

# Create a union of relevant passage IDs and sampled non-relevant passage IDs
final_passage_ids = relevant_passage_ids.union(sampled_non_relevant_passage_ids)
print(f"Total passage IDs after combining relevant and sampled non-relevant: {len(final_passage_ids)}")

Total passage IDs after combining relevant and sampled non-relevant: 209096


In [16]:
# Create a filtered passage collection dataframe with only the final passage IDs
filtered_passage_collection_df = passage_collection_df[
    passage_collection_df["passage_id"].isin(final_passage_ids)
]

In [17]:
# Load pre-trained embedding model
model = SentenceTransformer(
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
)

In [18]:
BATCH_SIZE = 64
PROCESSING_CHUNK_SIZE = 100000 # Process in batches to prevent GPU overloading
PASSAGES_WITH_EMBEDDINGS_OUTPUT_PATH = "../data/ms_marco/processed/passages_with_embeddings.jsonl"

# Save the passages with embeddings to a JSONL file
# Process in chunks to avoid memory issues
with open(PASSAGES_WITH_EMBEDDINGS_OUTPUT_PATH, "w") as f:
    total_rows = len(filtered_passage_collection_df)
    num_chunks = math.ceil(total_rows / PROCESSING_CHUNK_SIZE)

    print(f"Starting encoding for {total_rows} passages in {num_chunks} chunks...")

    for i in range(num_chunks):
        start_idx = i * PROCESSING_CHUNK_SIZE
        end_idx = min((i + 1) * PROCESSING_CHUNK_SIZE, total_rows) # ensure we don't go past end of DF

        print(f"Processing chunk {i + 1}/{num_chunks}, rows {start_idx} to {end_idx - 1}...")

        # Get the chunk from the DF
        chunk_df = filtered_passage_collection_df.iloc[start_idx:end_idx]
        # Get data for this chunk
        passages = chunk_df["passage_text"].tolist()
        passage_ids = chunk_df["passage_id"].tolist()
        # Generate passage embeddings for this chunk
        embeddings = model.encode(
            passages, 
            batch_size=BATCH_SIZE, 
            show_progress_bar=False, 
            convert_to_numpy=False
        )

        # Write this chunk's records to the file immediately
        for pid, passage, embedding in zip(passage_ids, passages, embeddings):
            # Build a list of dictionaries where each row is:
            # {"pid": "0", "passage": "some text", "embedding": [0.0312, -0.0249, ...]}
            record = {
                "pid": str(pid),
                "passage": passage,
                "embedding": embedding.tolist()  # Convert numpy array to list for JSON serialization
            }
            f.write(json.dumps(record) + "\n")
        

print(f"Passages with embeddings saved to {PASSAGES_WITH_EMBEDDINGS_OUTPUT_PATH}")

Starting encoding for 209096 passages in 3 chunks...
Processing chunk 1/3, rows 0 to 99999...
Processing chunk 2/3, rows 100000 to 199999...
Processing chunk 3/3, rows 200000 to 209095...
Passages with embeddings saved to ../data/ms_marco/processed/passages_with_embeddings.jsonl
